In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../outputs/models"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

df = pd.read_csv("../data/raw/ecommerce_customer_churn_large.csv")
df = df.drop(columns=["customer_id"])
df.shape

(200000, 10)

In [2]:
# Encode categorical variables
df["gender"] = df["gender"].map({"Female":0, "Male":1}).astype(int)
df["gender"].value_counts()

gender
0    100088
1     99912
Name: count, dtype: int64

In [3]:
# Encode subscription_type ordinal
tier_map = {"Basic":0, "Silver":1, "Gold":2, "Platinum":3}
df["subscription_rank"] = df["subscription_type"].map(tier_map).astype(int)
df[["subscription_rank","subscription_type"]].head()

,subscription_rank,subscription_type
0,0,Basic
1,0,Basic
2,3,Platinum
3,1,Silver
4,1,Silver


In [4]:
# Encode city and subscription
df = pd.get_dummies(df, columns=["city", "subscription_type"], drop_first=True, dtype=int)
df.shape

(200000, 18)

In [5]:
# VErify the encoded columns
print(df.columns.tolist())
df.head()

['age', 'gender', 'tenure_months', 'avg_order_value', 'total_orders', 'last_purchase_days_ago', 'support_tickets', 'churn', 'subscription_rank', 'city_Chennai', 'city_Delhi', 'city_Hyderabad', 'city_Kolkata', 'city_Mumbai', 'city_Pune', 'subscription_type_Gold', 'subscription_type_Platinum', 'subscription_type_Silver']


,age,gender,tenure_months,avg_order_value,total_orders,last_purchase_days_ago,support_tickets,churn,subscription_rank,city_Chennai,city_Delhi,city_Hyderabad,city_Kolkata,city_Mumbai,city_Pune,subscription_type_Gold,subscription_type_Platinum,subscription_type_Silver
0,56,0,82,7722,56,704,49,1,0,0,0,0,0,1,0,0,0,0
1,69,0,28,2127,248,357,39,0,0,0,0,0,1,0,0,0,0,0
2,46,1,98,2775,68,121,23,0,3,0,1,0,0,0,0,0,1,0
3,32,0,46,1698,309,39,47,0,1,0,0,0,0,0,1,0,0,1
4,60,0,90,6520,147,403,32,0,1,0,0,0,0,0,0,0,0,1


In [8]:
# Derived features

# Order Per Month
df["orders_per_month"]=df["total_orders"]/df["tenure_months"].clip(lower=1)
# Total revenue
df["total_revenue"]=df["avg_order_value"]*df["total_orders"]
# Revenue Per Month
df["revenue_per_month"]=df["total_revenue"]/df["tenure_months"].clip(lower=1)
# Tickets per Order
df["tickets_per_order"]=df["support_tickets"]/df["total_orders"].clip(lower=1)
# Dormant
df["is_dormant"]=(df["last_purchase_days_ago"]>180).astype(int)

df[["orders_per_month", "total_revenue", "revenue_per_month", "tickets_per_order", "is_dormant"]].describe()

,orders_per_month,total_revenue,revenue_per_month,tickets_per_order,is_dormant
count,200000.000000,2.000000e+05,2.000000e+05,200000.000000,200000.000000
mean,11.207669,1.275552e+06,5.713810e+04,0.337517,0.752080
std,31.641203,1.099366e+06,1.880328e+05,1.601921,0.431806
min,0.008403,2.200000e+02,1.864407e+00,0.000000,0.000000
25%,2.090909,3.694615e+05,6.641182e+03,0.047847,1.000000
50%,4.187500,9.617420e+05,1.827662e+04,0.097625,1.000000
75%,8.300000,1.936904e+06,4.231204e+04,0.195122,1.000000
max,499.000000,4.987505e+06,4.909661e+06,49.000000,1.000000


In [9]:
# Check NaN after division
new_features = ["orders_per_month", "revenue_per_month", "tickets_per_order"]
print("Inf count: ", np.isinf(df[new_features]).sum().sum())
print("NaN count: ", df[new_features].isna().sum().sum())

Inf count:  0
NaN count:  0


In [10]:
# SPlit X/y
y = df["churn"].copy()
X = df.drop(columns=["churn"])
print(X.shape)
print(y.shape)
print(y.mean().round(3))

(200000, 22)
(200000,)
0.372


In [11]:
numeric_to_scale = ["age", "tenure_months", "avg_order_value", "total_orders", 
                    "last_purchase_days_ago", "support_tickets","orders_per_month", "total_revenue",
                    "revenue_per_month", "tickets_per_order", "subscription_rank"]
print(f"Will scale {len(numeric_to_scale)} columns") #Sanity check

Will scale 11 columns


In [12]:
# Trial fit and tranform to test the Scaler
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_to_scale] = scaler.fit_transform(X[numeric_to_scale])
X_scaled[numeric_to_scale].describe().round(2)

,age,tenure_months,avg_order_value,total_orders,last_purchase_days_ago,support_tickets,orders_per_month,total_revenue,revenue_per_month,tickets_per_order,subscription_rank
count,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.00,200000.0
mean,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.0
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.0
min,-1.70,-1.71,-1.73,-1.73,-1.73,-1.69,-0.35,-1.16,-0.30,-0.21,-1.0
25%,-0.83,-0.87,-0.87,-0.87,-0.87,-0.86,-0.29,-0.82,-0.27,-0.18,-1.0
50%,-0.03,0.01,-0.00,0.00,-0.00,-0.03,-0.22,-0.29,-0.21,-0.15,0.0
75%,0.84,0.88,0.86,0.87,0.87,0.87,-0.09,0.60,-0.08,-0.09,1.0
max,1.70,1.72,1.73,1.73,1.73,1.70,15.42,3.38,25.81,30.38,2.0


In [13]:
joblib.dump(scaler, f"{MODELS_DIR}/scaler.joblib")
print("Scaler saved.")

Scaler saved.


In [14]:
# Unscaled version (for tree models)
X.assign(churn=y).to_csv(f"{PROCESSED_DIR}/churn_features_unscaled.csv", index=False)

# Scaled version (for LogReg)
X_scaled.assign(churn=y).to_csv(f"{PROCESSED_DIR}/churn_features_scaled.csv", index=False)

In [16]:
print("Dtypes:")
print(X.dtypes.value_counts())
print()
print(f"Any NaN: {X.isna().any().any()}")
print(f"Any Inf: {np.isinf(X.select_dtypes(include=np.number)).any().any()}")
print()
print("Feature matrix preview:")
X.head(3)

Dtypes:
int64      19
float64     3
Name: count, dtype: int64

Any NaN: False
Any Inf: False

Feature matrix preview:


,age,gender,tenure_months,avg_order_value,total_orders,last_purchase_days_ago,support_tickets,subscription_rank,city_Chennai,city_Delhi,...,city_Mumbai,city_Pune,subscription_type_Gold,subscription_type_Platinum,subscription_type_Silver,orders_per_month,total_revenue,revenue_per_month,tickets_per_order,is_dormant
0,56,0,82,7722,56,704,49,0,0,0,...,1,0,0,0,0,0.682927,432432,5273.560976,0.875000,1
1,69,0,28,2127,248,357,39,0,0,0,...,0,0,0,0,0,8.857143,527496,18839.142857,0.157258,1
2,46,1,98,2775,68,121,23,3,0,1,...,0,0,0,1,0,0.693878,188700,1925.510204,0.338235,0
